create Dataframe from storage (files / dir ) To read the delimited data from any storage (dbfs , hdfs , lfs , cloud staorgaes )

spark.read.csv opition -> dataframe

csv is the built in source

Deafults
spark.read.csv

default options :

header = False

default cols = _c0 , _c1 _c2

delimiter = ","

default data type for all columns = string

In [0]:
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/") # file path


In [0]:
%fs head dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/emp.csv

In [0]:
print(type(custdata)) # dataframe
custdata.show() # similar to collect -> action )
# show action , display default 20 records 

In [0]:
# describe table 
custdata.printSchema()

In [0]:
# view few or more records 
custdata.show(10,False)
#custdata.show(200)

In [0]:

display(custdata)

In [0]:
# changing the column names from _c0, _c1 to actuals 
#Since header is there, we can remove .toDF
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/").toDF("custid","name")

custdata.show(2) # display only 2 records 

custdata.printSchema()

In [0]:
#Since header is there, we can remove .toDF
#supose we recivied a file with header
#column names we need to pick from the header
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/",header=True)

custdata.show(2) # display only 2 records 

custdata.printSchema()

In [0]:
# how to get the number of records in the dataframe
# select count(1) from table 
# count is an action , its return integer result , trigger execution , job is created
custdata.count()

In [0]:
# data type for all columns treated default as string
# based on the data we have generate the schema with proper data type 
# performance if we read large data inferSchema is not a good option 
custdata=spark.read.csv("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/",header=True,inferSchema=True)

custdata.show(5)

custdata.printSchema()

In [0]:
#Different delimited (|)

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/raw/emp1.csv

In [0]:
emp_df=spark.read.csv(path="dbfs:/Volumes/izwd37dev/wd37db/rawdatta/raw/emp1.csv",header=True,inferSchema=True,sep="|")

In [0]:
emp_df.show()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/raw/emp1.csv

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/raw/emp2.csv

In [0]:

# options , way to create dataframe using csv with different options 
custdata=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/raw/",header=True,sep="|") # directory 
custdata.show(10)
custdata.count()

In [0]:
# creating a datfrme from list of paths 
custdata=spark.read.csv(path=["/Volumes/izwd37dev/wd37db/rawdatta/raw/emp1.csv","/Volumes/izwd37dev/wd37db/rawdatta/raw/emp2.csv"],header=True,sep="|") # multiple files  
custdata.show(10)
custdata.count()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales_chennai.csv

In [0]:
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai/sales*",header=True,inferSchema=True)
sales_df.show(5)
print(sales_df.count())
sales_df.printSchema()

In [0]:
# recursiveFileLookup - read all sub dir
# pathGlobFilter - apply the filter / pattern globally on all dir /sub dir 
sales_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/salesdata",header=True,inferSchema=True,recursiveFileLookup=True,pathGlobFilter="sales*")
sales_df.show(5)
sales_df.printSchema()

In [0]:

%fs ls /Volumes/izwd37dev/wd37db/rawdatta/salesdata/chennai

In [0]:
# inferSchema -> generating schema by reading the entire data 
# to avoid reading the data for genrating schema , - performance issue 
# when we are going with inferschema its more dynamic  - dq issue 

emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,inferSchema=True)
emp_df.show()
emp_df.printSchema() 

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/csv/custs

In [0]:
#we can create our own schema and apply while reading data
'''
create table cust(custid int,fname string,lname string,age int , prof string)
'''
cust_schema="custid int,fname string,lname string,age int , prof string"
custdata=spark.read.csv(path="/Volumes/izwd37dev/wd37db/rawdatta/csv/custs",schema=cust_schema)   
custdata.show(5,False)
custdata.count()
custdata.printSchema()

In [0]:
# programmatically define the schema - using StructType and StructField

from pyspark.sql.types import StructType,StructField,IntegerType,StringType

'''
int -> integrType
str -> StringType

StructType - structure (row) / record 
StructField - column -> [name , type ]


'''
schema=StructType([
    StructField("eid",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("age",IntegerType(),True)])
    
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp1.csv",sep="|",header=True,schema=schema)
emp_df.show()
emp_df.printSchema() 

In [0]:
#built in Source
#CSV
#JSON - Serialized form of data. its semi structured key value data. Along with data it will keep field name as well.
        #Key is column and value is data
        #inferSchema by default true,sep,header not available for json
#XML
#parquet
#delta
#ORC
#Excel
#jdbc
#table

In [0]:

%fs head /Volumes/izwd37dev/wd37db/rawdatta/json/emp.json

In [0]:
emp_schema="id int,name string,age int"
emp_json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_schema)
emp_json_df.show()
emp_json_df.printSchema()

In [0]:
# connect external source 
# genric way to read data using spark

# spark.read.option("k","v").format("source").load()

# csv 

# df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs") --Usual way


cust_df=spark.read.option("header","True").option("inferSchema","True").option("delimiter",",").format("csv").load("/Volumes/izwd37dev/wd37db/rawdatta/custs_header") #Its generic. you can use json,xml etc

cust_df.show()

In [0]:
emp_scehma="id int,name string,age int"
#emp_json_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)

emp_json_df=spark.read.schema(emp_scehma).format("json").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")

emp_json_df.show()
emp_json_df.printSchema()

In [0]:
emp_scehma="id int,name string,age int"
#emp_json_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json",schema=emp_scehma)

emp_json_df=spark.read.schema(emp_scehma).format("json").load("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/json/emp.json")

emp_json_df.show()
emp_json_df.printSchema()

In [0]:
# create data frame from json 


cust_df=spark.read.json("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custjson/")
cust_df.show(5)
cust_df.printSchema()

In [0]:
# create dataframe from xml
# Each row you can identify with tag called customer. Ex: rowTag="customer"

cust_df=spark.read.xml("dbfs:/Volumes/izwd37dev/wd37db/rawdatta/custxml/",rowTag="customer")
cust_df.show(5)
cust_df.printSchema()
#

In [0]:
#bad record (not matching the schema )
#allow - By default
#fail
#ignore
#option -> mode

#permissive -> permitting , RCA later
#dropmalformed - blindly drop that record proceed
#failfast - immediately fail the whole job

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema)
stud_df.show()
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE")
stud_df.show()
stud_df.printSchema()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="dropMalformed")
stud_df.show()

In [0]:
stud_schema="sid int,sname string,dept string,yearofbirth int"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="failFast")
stud_df.show()

In [0]:

stud_schema="sid int,sname string,dept string,yearofbirth int,error_rec string"
stud_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student/stud_dept.csv",header=True,schema=stud_schema,mode="PERMISSIVE",columnNameOfCorruptRecord="error_rec")
stud_df.show()
stud_df.printSchema()

In [0]:
#reading a comma seprated employee data (eid,ename,address)

#address colum,n also have comma

#source sending header ifo / footer

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp3.csv

In [0]:
emp_schema="eid int,ename string,address string"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp3.csv",sep=",",header=True,schema=emp_schema)
emp_df.show()

In [0]:
#we have value as 'velacherry,chennai' but in above we are getting as velacherry alone. to resolve use below
#comment="#" - Dont consider if commented
#Note: Spark is looking for the 1st cama,if found that will be 1st column like wise it will do for the rest of the rows
#quote="'" => Any column which has quote right within the value,dont consider , as delimiter. Treat it as single column
#escape="~" => If your value has 's like special character dont conider as a quote charecter. That is a value
#Note - raja's street,mumbai,88 - 's is not a quote character.Its a value

emp_schema="eid int,ename string,address string"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp3.csv",sep=",",header=True,schema=emp_schema,quote="'",comment="#")
emp_df.show(10,False) # show default first 20 records 
emp_df.printSchema() 

In [0]:
#escape="~" => If your value has 's like special character dont conider as a quote charecter. That is a value
# In that case source will send with escape charater
emp_schema="eid int,ename string,address string"
emp_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/emp3.csv",sep=",",header=True,schema=emp_schema,quote="'",comment="#",escape="~")
emp_df.show(10,False) # show default first 20 records 
emp_df.printSchema() 

In [0]:
emp_df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/emp3_output",header=True,sep="|",quote="'",escape="~")

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp3_output/part-00000-tid-7526742694117355360-7422e414-7eb2-49b2-9241-22cbd5bbaae2-278-1-c000.csv

In [0]:
#create dataframe from python objects
#programtically create dataframe

In [0]:
data=[(1,"jaya,anu",23),(2,"ram,shyam",25),(3,"sita,geeta",27)]
col_name=["id","name","age"]

df=spark.createDataFrame(data,col_name)
df.show()
df.printSchema()

In [0]:
df.write.mode("overwrite").csv("/Volumes/izwd37dev/wd37db/rawdatta/pythondfout",header=True,sep=",",quote="'")

In [0]:
#without  quote="'"
df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/pythondfout",header=True,sep=",")
df1.show()

In [0]:
#with  quote="'"
df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/pythondfout",header=True,sep=",",quote="'")
df1.show()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp2.json

In [0]:
# Json options explore
# yyyy-MM-dd - spark default date format.If the data is in this format. spark will read 
#single row json means every row will get terminated in single line.
#Ex:{"id":100,"name":"patel","dob":"21/01/1996"}
    #{"id":101,"name":"patel1","dob":"21/01/1997"}

#multi row json means every row will get terminated in multiple line.
     #{
    #"id":100,
   # "name":"patel",
   # "dob":"21/01/1996"
  #  }

#to
json_ddl_schema="id int,name string,dob date"
e_json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/emp2.json",multiLine=True,schema=json_ddl_schema)
e_json_df.show()
e_json_df.printSchema()

In [0]:
#Source format we are giving here
e_json_df=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/emp2.json",multiLine=True,schema=json_ddl_schema,dateFormat="dd/MM/yyyy")
e_json_df.show()
e_json_df.printSchema()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/emp3.json
#here id column without "" is there.its not a json format

In [0]:
#dbfs is the URI
#Volume is the databricks object. under DBFS URI we are accessing the volume
#Ex - dbfs-Its nothing but abstraction layer on top of your cloud storage. Databricks perspective its volume, internally its pointing to cloud storage. DBFS is the URI, volume is the folder

In [0]:
#allowUnquotedFieldNames => lets say for id column if we dont have "", we can enable this option. but this is for json alone
                            #  {
                            # id:100,
                            # "name":"patel",
                            # "dob":"21/01/1996"
                            #  }

#allowSingleQuotes=True => If the value has quote,we can enable this option. but this is for json alone

json_ddl_schema="id int,name string,dob date"
e_json_df1=spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/emp5.json",multiLine=True,schema=json_ddl_schema,columnNameOfCorruptRecord="error_rec",allowUnquotedFieldNames=True,allowSingleQuotes=True,dateFormat="dd/MM/yyyy")
e_json_df1.show()